# DIA-NN peptide and protein export

Exports unnormalized, per-run peptide and protein tables from one DIA-NN main report (`.parquet`).

1. Edit the settings in cell 1 (at least `input_file`).
2. Run the cells from top to bottom with **Shift+Enter**.

Decoys and contaminant protein groups (`contaminant_`, `cRAP-`) are removed before the q-value cutoffs are applied. The code lives in the `diann_tools` folder next to this notebook; see `README.md` for the output columns.

Tables are written to the `results` folder of the repository, which Git ignores. Before committing changes to this notebook, clear its outputs (**Kernel → Restart Kernel and Clear Outputs of All Cells**).

In [ ]:
# 1. Settings: edit these lines.
from pathlib import Path

input_file = Path(r"D:\UNT\Data\DIA_20min.parquet")
# Results go to the repository's results/ folder, which Git ignores.
output_dir = Path.cwd().parent / "results" / f"{input_file.stem}_tables"   # must be a new folder

# Q-value cutoffs (inclusive). Set a value to None to switch that filter off.
q_cutoffs = {
    "Q.Value": 0.01,            # run-specific precursor
    "Global.Q.Value": 0.01,     # experiment-wide precursor
    "Global.PG.Q.Value": 0.01,  # experiment-wide protein group
    "PG.Q.Value": 0.05,         # run-specific protein group (protein table only)
    "Lib.Q.Value": None,
    "Lib.PG.Q.Value": None,
}

# Protein intensity: "auto", "reported" (DIA-NN PG.Quantity) or "sum-proteotypic"
protein_quantity = "auto"


In [ ]:
# 2. Check which runs the report contains before exporting.
from diann_tools.reader import count_rows_per_run

count_rows_per_run(input_file)


In [ ]:
# 3. Export the tables.
# To rerun, choose a new output_dir in cell 1. After editing a file in
# diann_tools, restart the kernel so the change is picked up.
from diann_tools import export_report

run_summary = export_report(input_file, output_dir, q_cutoffs, protein_quantity)
run_summary[[
    "Run", "Contaminant.Rows.Removed", "Peptide.Input.Precursors", "Peptides",
    "Unique.Stripped.Peptides", "Protein.Groups", "MS1.PPM.Median", "MS2.Best.Fragment.PPM.Median",
]]


In [ ]:
# 4. Open the tables of one run (1 = first row of the summary above).
import pandas as pd

run_number = 1
selected = run_summary.iloc[run_number - 1]
proteins = pd.read_csv(output_dir / selected["Proteins.File"], sep="\t")
peptides = pd.read_csv(output_dir / selected["Peptides.File"], sep="\t")

print(selected["Run"])
display(proteins.head())
display(peptides.head())
